Loading the dataset

In [ ]:
!wget 'https://zenodo.org/records/7119953/files/clue.zip?download=1' -O clue.zip

--2026-05-27 16:41:53--  https://zenodo.org/records/7119953/files/clue.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.153.219, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635105552 (606M) [application/octet-stream]
Saving to: ‘clue.zip’

clue.zip            100%[===================>] 605.68M  18.5MB/s    in 11m 19s 

2026-05-27 16:53:13 (913 KB/s) - ‘clue.zip’ saved [635105552/635105552]



Unzipping dataset

In [ ]:
!unzip clue.zip

Archive:  clue.zip
  inflating: clue.json               


seeing the directory files

In [ ]:
!ls

clue.json  clue.zip  sample_data


Calculating lines for 1GB file

In [ ]:
import json

sample = []
with open('clue.json', 'r') as f:
    for i, line in enumerate(f):
        if i >= 10:  # Read only first 10 lines for sampling
            break
        try:
            sample.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line {i+1}: {e}")
            continue

if sample:
    record_size = sum(len(json.dumps(r)) for r in sample) // len(sample)
    print(f"Approx average record size: {record_size / 1024:.2f} KB")
else:
    print("No valid JSON records found in the sample.")

Approx average record size: 0.29 KB


Building the 1GB file

In [ ]:
import json

target_bytes = 1_073_741_824  # 1GB in bytes
record_size = 297  # Based on your average estimate

max_records = target_bytes // record_size
print("Sampling", max_records, "records to approach 1GB.")

in_path = 'clue.json'               # input large file
out_path = 'clue_sample_1GB.jsonl'  # output sampled file

count, total_bytes = 0, 0
with open(in_path, 'r') as fin, open(out_path, 'w') as fout:
    for line in fin:
        if count >= max_records or total_bytes > target_bytes:
            break
        fout.write(line)
        count += 1
        total_bytes += len(line.encode('utf-8'))

print(f"Wrote {count} records, approx {total_bytes/1024/1024:.2f} MB")

Sampling 3615292 records to approach 1GB.
Wrote 3479163 records, approx 1024.00 MB


In [ ]:
!ls

clue.json  clue_sample_1GB.jsonl  clue.zip  sample_data


In [ ]:
from google.colab import files
files.download('clue_sample_1GB.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
with open('clue_sample_1GB.jsonl', "r", encoding="utf-8") as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())

{"params": {"path": "/useful-turquoise-cardinal-historian"}, "type": "file_accessed", "time": "2017-07-07T08:57:57Z", "uid": "yellow-chocolate-carp-busdriver", "id": 1, "uidType": "name"}
{"params": {"path": "/useful-turquoise-cardinal-historian", "run": true}, "type": "file_updated", "time": "2017-07-07T08:58:02Z", "uid": "yellow-chocolate-carp-busdriver", "id": 2, "uidType": "name"}
{"params": {"path": "/useful-turquoise-cardinal-historian", "run": true}, "type": "file_written", "time": "2017-07-07T08:58:02Z", "uid": "yellow-chocolate-carp-busdriver", "id": 3, "uidType": "name"}
{"params": {"path": "/chinese-teal-meerkat-garagemanager/filthy-fuchsia-flamingo-biochemist/leading-emerald-cattle-planningmanager/outer-lavender-lion-hosierymechanic/eventual-violet-barnacle-insuranceassessor", "run": true}, "type": "file_created", "time": "2017-07-07T08:59:11Z", "uid": "renewed-emerald-hare-productionplanner", "id": 4, "uidType": "name"}
{"params": {"path": "/chinese-teal-meerkat-garagemana

In [ ]:
with open('clue_sample_1GB.jsonl', "r", encoding="utf-8") as f:
    lines = f.readlines()
    for line in lines[-10:]:
        print(line.rstrip())

{"params": {"trigger": 2, "path": "/chosen-blue-marmoset-metallurgist/bright-amethyst-hoverfly-zookeeper"}, "type": "version_deleted", "time": "2017-10-03T22:10:11Z", "uid": "shared-fuchsia-cardinal-buildingadvisor", "id": 3479154, "uidType": "name"}
{"params": {"trigger": 2, "path": "/chosen-blue-marmoset-metallurgist/satisfactory-tomato-turkey-hospitaltechnician"}, "type": "version_deleted", "time": "2017-10-03T22:10:11Z", "uid": "shared-fuchsia-cardinal-buildingadvisor", "id": 3479155, "uidType": "name"}
{"params": {"trigger": 2, "path": "/chosen-blue-marmoset-metallurgist/planned-turquoise-bonobo-hospitalconsultant"}, "type": "version_deleted", "time": "2017-10-03T22:10:11Z", "uid": "shared-fuchsia-cardinal-buildingadvisor", "id": 3479156, "uidType": "name"}
{"params": {"trigger": 2, "path": "/chosen-blue-marmoset-metallurgist/forthcoming-black-stingray-goodshandler"}, "type": "version_deleted", "time": "2017-10-03T22:10:11Z", "uid": "shared-fuchsia-cardinal-buildingadvisor", "id":

Searching for top active users (using a scoring system)

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

# ---- PARAMETERS ----
FILE = "clue_sample_1GB.jsonl"
DATA_START = pd.Timestamp("2017-07-07")
DATA_END = pd.Timestamp("2017-10-03")  # inclusive
MIN_ACTIVE_DAYS = 75
MIN_MEDIAN_EVENTS = 50
MAX_BURST_RATIO = 20
TOP_N = 20
CHUNKSIZE = 100_000

print("Stage 1: Aggregating user-day statistics (in chunks)...")

# ---- ACCUMULATORS ----
# agg[uid][date] = counter of event types for that uid/date
per_user_per_day_events = defaultdict(Counter)

for chunk in pd.read_json(FILE, lines=True, chunksize=CHUNKSIZE):
    chunk['date'] = pd.to_datetime(chunk['time']).dt.date  # Extract just calendar date
    # Optional: make sure 'uid', 'type', etc. exist
    if not set(['uid','type','id','date']).issubset(chunk.columns):
        raise ValueError("Chunk missing required columns.")
    # Group per (uid, date), count events and event types
    for (uid, date), group in chunk.groupby(['uid', 'date']):
        per_user_per_day_events[uid][date] += len(group)

print("Done aggregating. Assembling summary table...")

# Convert accumulated counts to a DataFrame for per-UID stats
stats_records = []
for uid, date_counts in per_user_per_day_events.items():
    days = sorted(date_counts.keys())
    daily_counts = [date_counts[d] for d in days]
    total_events = sum(daily_counts)
    active_days = len(days)
    median_events = int(np.median(daily_counts)) if daily_counts else 0
    p95_events = int(np.percentile(daily_counts, 95)) if daily_counts else 0
    burst_ratio = p95_events / max(1, median_events)
    stats_records.append({
        'uid': uid,
        'total_events': total_events,
        'active_days': active_days,
        'median_events_per_active_day': median_events,
        'p95_events_per_active_day': p95_events,
        'burst_ratio': burst_ratio
    })

stats = pd.DataFrame(stats_records)

total_days = (DATA_END - DATA_START).days + 1  # ≈89
stats['coverage'] = stats['active_days'] / total_days

# ---- FILTERING & RANKING ----
filtered = stats[
    (stats['active_days'] >= MIN_ACTIVE_DAYS) &
    (stats['median_events_per_active_day'] >= MIN_MEDIAN_EVENTS) &
    (stats['burst_ratio'] <= MAX_BURST_RATIO)
].copy()

filtered['score'] = (
    np.log1p(filtered['total_events']) +
    2 * np.log1p(filtered['active_days']) +
    np.log1p(filtered['median_events_per_active_day'])
)
filtered = filtered.sort_values('score', ascending=False)

# ---- OUTPUT ----
shortlist = filtered.head(TOP_N)[['uid', 'total_events', 'active_days', 'coverage',
                                 'median_events_per_active_day', 'p95_events_per_active_day',
                                 'burst_ratio', 'score']]
print("\nTop active users (by score):\n")
print(shortlist)

shortlist.to_csv("top_active_users_chunked.csv", index=False)
print("\nSaved: top_active_users_chunked.csv")

# (OPTIONAL) If you also need unique_types, you can add similar logic using per-user-type counters.

Stage 1: Aggregating user-day statistics (in chunks)...
Done aggregating. Assembling summary table...

Top active users (by score):

                                         uid  total_events  active_days  \
47  statutory-silver-ocelot-warehousemanager        373454           89   
37          nice-orange-centipede-taxidriver        248343           89   
43   shared-fuchsia-cardinal-buildingadvisor        202615           89   
29              little-amber-minnow-reporter        108103           89   
31            logical-coral-pig-warehouseman         83654           89   
28           labour-crimson-donkey-golfcaddy         67558           89   
22                   icy-amaranth-cat-potter         54738           80   
41    renewed-emerald-hare-productionplanner         45603           81   
10  developed-harlequin-falcon-radiodirector         49985           88   
25   interim-coffee-gazelle-lightingdesigner         15506           89   
40       ready-silver-angelfish-quarrywork

In [ ]:
import pandas as pd
df_top_users = pd.read_csv('top_active_users_chunked.csv')
display(df_top_users.head())

,uid,total_events,active_days,coverage,median_events_per_active_day,p95_events_per_active_day,burst_ratio,score
0,statutory-silver-ocelot-warehousemanager,373454,89,1.0,4236,4292,1.013220,30.181783
1,nice-orange-centipede-taxidriver,248343,89,1.0,1429,7784,5.447166,28.687619
2,shared-fuchsia-cardinal-buildingadvisor,202615,89,1.0,1480,8852,5.981081,28.519160
3,little-amber-minnow-reporter,108103,89,1.0,1149,2085,1.814621,27.637986
4,logical-coral-pig-warehouseman,83654,89,1.0,1057,1137,1.075686,27.298211


Top 20 active users and their missing data

In [ ]:
from typing import Dict, Iterable

# -------------------
# CONFIG
# -------------------
JSONL_FILE = "clue_sample_1GB.jsonl"
CHUNKSIZE = 100_000

# Your shortlist from the activity-based selection stage
ACTIVE_USERS_FILE = "top_active_users_chunked.csv"

# Optional: restrict to only the top K active users to speed up
TOP_K_ACTIVE = None  # e.g., 50, or None to keep all rows in the CSV

# Fields we want to be present as often as possible.
# We score missingness for these.
# Notes:
# - "time", "uid", "type", "id" are expected always present; we don't score them.
# - params/location are dict-like. We'll handle them carefully.
FIELDS = [
    "uidType",
    "params",
    "isLocalIP",
    "role",
    "location",
]

# Nested location subfields (only meaningful when location is present)
LOCATION_SUBFIELDS = [
    "country",
    "city",
    "latitude",
    "longitude",
]

# Weights: increase weight for fields you care more about.
# If you "want most features", reasonable default is to weight all equally,
# but you can bump location.* because it's often missing and very valuable.
WEIGHTS = {
    "uidType": 1.0,
    "params": 1.0,
    "isLocalIP": 1.0,
    "role": 1.0,
    "location": 1.0,
    # nested fields
    "location.country": 1.0,
    "location.city": 0.5,
    "location.latitude": 0.5,
    "location.longitude": 0.5,
}

# -------------------
# HELPERS
# -------------------
def add_counts(target: Dict[str, int], source: Iterable):
    for k, v in source.items():
        target[k] += int(v)

def extract_location_series(loc_col: pd.Series, key: str) -> pd.Series:
    """
    loc_col is expected to contain dicts or NaN.
    Returns a Series where missing is NaN.
    """
    # loc_col may contain dicts; convert to python objects and extract key
    return loc_col.apply(lambda x: (x.get(key) if isinstance(x, dict) else None))

# -------------------
# LOAD ACTIVE USERS
# -------------------
active_df = pd.read_csv(ACTIVE_USERS_FILE)
if TOP_K_ACTIVE is not None:
    active_df = active_df.head(TOP_K_ACTIVE)
active_users = set(active_df["uid"].astype(str).tolist())

print(f"Loaded {len(active_users)} active candidate users from {ACTIVE_USERS_FILE}")

# -------------------
# ACCUMULATORS
# -------------------
total_events = defaultdict(int)

missing_counts = defaultdict(lambda: defaultdict(int))  # missing_counts[field][uid] = count_missing

present_counts = defaultdict(lambda: defaultdict(int))  # present_counts[field][uid] = count_present
# present_counts helps when you want to compute "coverage" and debug.

# -------------------
# STREAM JSONL
# -------------------
print("Streaming JSONL and computing missingness (chunked)...")

for chunk in pd.read_json(JSONL_FILE, lines=True, chunksize=CHUNKSIZE):
    # Restrict to candidate users early for speed
    if "uid" not in chunk.columns:
        raise ValueError("Missing required column 'uid' in the JSONL.")

    chunk["uid"] = chunk["uid"].astype(str)
    chunk = chunk[chunk["uid"].isin(active_users)]
    if chunk.empty:
        continue

    # Total events per uid
    vc = chunk["uid"].value_counts()
    add_counts(total_events, vc)

    # For each top-level field, count missing vs present
    for f in FIELDS:
        if f not in chunk.columns:
            # Column absent => treat all as missing for that chunk
            for uid, n in vc.items():
                missing_counts[f][uid] += int(n)
            continue

        is_missing = chunk[f].isna()

        miss_vc = chunk.loc[is_missing, "uid"].value_counts()
        pres_vc = chunk.loc[~is_missing, "uid"].value_counts()

        add_counts(missing_counts[f], miss_vc)
        add_counts(present_counts[f], pres_vc)

    # Nested location.* fields: only evaluate when location exists
    if "location" in chunk.columns:
        loc_present_mask = chunk["location"].apply(lambda x: isinstance(x, dict))
        # For users with location missing (not dict), those subfields are missing by definition
        # but we separate scoring into:
        # - location missing (top-level)
        # - and subfield missing given location present (quality of location data)

        loc_present = chunk.loc[loc_present_mask, ["uid", "location"]]
        if not loc_present.empty:
            for sub in LOCATION_SUBFIELDS:
                series = extract_location_series(loc_present["location"], sub)
                sub_missing = series.isna()
                miss_vc = loc_present.loc[sub_missing, "uid"].value_counts()
                pres_vc = loc_present.loc[~sub_missing, "uid"].value_counts()

                field_name = f"location.{sub}"
                add_counts(missing_counts[field_name], miss_vc)
                add_counts(present_counts[field_name], pres_vc)

print("Done. Building ranked table...")

# -------------------
# BUILD OUTPUT TABLE
# -------------------
rows = []
for uid, n_total in total_events.items():
    row = {"uid": uid, "total_events": int(n_total)}
    # coverage per field
    for f in list(FIELDS) + [f"location.{s}" for s in LOCATION_SUBFIELDS]:
        miss = missing_counts[f].get(uid, 0)
        pres = present_counts[f].get(uid, 0)

        # For top-level fields, pres+miss should be close to total_events (unless some values are non-null but not counted)
        # For location.* subfields, pres+miss counts only events where location was present.
        denom = (pres + miss)
        rate = (miss / denom) if denom > 0 else 1.0

        row[f"{f}_missing"] = int(miss)
        row[f"{f}_present"] = int(pres)
        row[f"{f}_missing_rate"] = float(rate)

    # Weighted missingness score: lower is better
    score = 0.0
    weight_sum = 0.0
    for f, w in WEIGHTS.items():
        col = f"{f}_missing_rate"
        if col in row:
            score += w * row[col]
            weight_sum += w

    row["missingness_score"] = score / weight_sum if weight_sum > 0 else 1.0
    rows.append(row)

ranked = pd.DataFrame(rows)

# Merge activity stats so you can see volume/coverage next to missingness
ranked = ranked.merge(active_df, on="uid", how="left")

# Sort: least missingness first; tie-break by higher activity/coverage
ranked = ranked.sort_values(
    ["missingness_score", "active_days", "total_events_y"],
    ascending=[True, False, False]
)

# Clean up duplicate column names from merge
# total_events_x = recomputed from jsonl filtered, total_events_y = from active summary
ranked = ranked.rename(columns={"total_events_x": "total_events_in_jsonl_filtered",
                                "total_events_y": "total_events_summary"})

ranked.to_csv("active_users_ranked_by_missingness.csv", index=False)
print("Saved: active_users_ranked_by_missingness.csv")

print("\nTop 20 active users with least missingness:\n")
cols_to_show = [
    "uid",
    "missingness_score",
    "active_days",
    "coverage",
    "median_events_per_active_day",
    "burst_ratio",
    "role_missing_rate",
    "isLocalIP_missing_rate",
    "location_missing_rate",
    "location.country_missing_rate",
    "params_missing_rate",
]
cols_to_show = [c for c in cols_to_show if c in ranked.columns]
print(ranked[cols_to_show].head(20).to_string(index=False))

Loaded 14 active candidate users from top_active_users_chunked.csv
Streaming JSONL and computing missingness (chunked)...
Done. Building ranked table...
Saved: active_users_ranked_by_missingness.csv

Top 20 active users with least missingness:

                                     uid  missingness_score  active_days  coverage  median_events_per_active_day  burst_ratio  role_missing_rate  isLocalIP_missing_rate  location_missing_rate  location.country_missing_rate  params_missing_rate
            little-amber-minnow-reporter           0.600000           89  1.000000                          1149     1.814621                0.0                     1.0                    1.0                            1.0                  0.0
      spectacular-copper-cheetah-postman           0.600000           88  0.988764                            72     2.097222                0.0                     1.0                    1.0                            1.0                  0.0
           central-silv

Final ranked list of users (for a consistent quantitative study)

In [ ]:
print(ranked[cols_to_show].head(50).to_string(index=False))

                                     uid  missingness_score  active_days  coverage  median_events_per_active_day  burst_ratio  role_missing_rate  isLocalIP_missing_rate  location_missing_rate  location.country_missing_rate  params_missing_rate
            little-amber-minnow-reporter           0.600000           89  1.000000                          1149     1.814621                0.0                     1.0                    1.0                            1.0                  0.0
      spectacular-copper-cheetah-postman           0.600000           88  0.988764                            72     2.097222                0.0                     1.0                    1.0                            1.0                  0.0
           central-silver-slug-zookeeper           0.600000           84  0.943820                            64     4.484375                0.0                     1.0                    1.0                            1.0                  0.0
statutory-silver-ocelot-

Checking for overall missing data (for feature engineering)

In [ ]:

FILE = "clue_sample_1GB.jsonl"
CHUNKSIZE = 200_000
FIELDS = ["role", "isLocalIP", "location", "uidType", "params"]

total = 0
present = {f: 0 for f in FIELDS}

for chunk in pd.read_json(FILE, lines=True, chunksize=CHUNKSIZE):
    total += len(chunk)
    for f in FIELDS:
        if f in chunk.columns:
            present[f] += chunk[f].notna().sum()

print("Total events checked:", total)
for f in FIELDS:
    print(f"{f}: present {present[f]} ({present[f]/total:.4%})")

Total events checked: 3479163
role: present 956640 (27.4963%)
isLocalIP: present 15723 (0.4519%)
location: present 4640 (0.1334%)
uidType: present 3479163 (100.0000%)
params: present 3479163 (100.0000%)


Extracting JSON lines of the best presumed user over the first 3 months

In [ ]:
JSONL_FILE = "clue_sample_1GB.jsonl"
UID = "spectacular-copper-cheetah-postman"

# Optional filters (set to None to disable)
START_DATE = None          # "2017-07-07"
END_DATE = None            # "2017-10-03"
MAX_LINES = 20             # how many matching events to print

CHUNKSIZE = 200_000

printed = 0

for chunk in pd.read_json(JSONL_FILE, lines=True, chunksize=CHUNKSIZE):
    # Ensure uid exists and compare as string
    if "uid" not in chunk.columns:
        raise ValueError("Missing 'uid' in JSONL")
    chunk["uid"] = chunk["uid"].astype(str)

    sub = chunk[chunk["uid"] == UID]
    if sub.empty:
        continue

    # Optional date filtering (expects a 'time' column)
    if (START_DATE or END_DATE):
        if "time" not in sub.columns:
            raise ValueError("START_DATE/END_DATE provided but column 'time' not found")

        # Convert to datetime once for the filtered subframe
        t = pd.to_datetime(sub["time"], errors="coerce", utc=True)

        if START_DATE:
            sub = sub[t >= pd.Timestamp(START_DATE, tz="UTC")]
        if END_DATE:
            # inclusive end date: add 1 day and use <
            sub = sub[t < (pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1))]

        if sub.empty:
            continue

    # Print rows as JSON (one per line)
    for _, row in sub.iterrows():
        print(row.to_json())
        printed += 1
        if printed >= MAX_LINES:
            break

    if printed >= MAX_LINES:
        break

print(f"\nPrinted {printed} lines for uid={UID}")

{"params":{"path":"\/chinese-teal-meerkat-garagemanager\/surrounding-maroon-nightingale-securitycontroller\/flat-amaranth-chameleon-ticketagent\/productive-amber-spoonbill-joiner\/thoughtless-crimson-goose-heatingengineer"},"type":"file_accessed","time":"2017-07-07T09:34:18Z","uid":"spectacular-copper-cheetah-postman","id":203,"uidType":"name","role":"technical","isLocalIP":null}
{"params":{"path":"\/chinese-teal-meerkat-garagemanager\/surrounding-maroon-nightingale-securitycontroller\/net-azure-sloth-printfinisher\/official-crimson-mole-pilot"},"type":"file_accessed","time":"2017-07-07T09:40:47Z","uid":"spectacular-copper-cheetah-postman","id":231,"uidType":"name","role":"technical","isLocalIP":null}
{"params":{"path":"\/chinese-teal-meerkat-garagemanager\/surrounding-maroon-nightingale-securitycontroller\/net-azure-sloth-printfinisher\/official-crimson-mole-pilot"},"type":"file_accessed","time":"2017-07-07T09:41:47Z","uid":"spectacular-copper-cheetah-postman","id":251,"uidType":"name

Building the new features (per day statistics)

In [ ]:
import pandas as pd

JSONL_FILE = "clue_sample_1GB.jsonl"
UID = "spectacular-copper-cheetah-postman"
CHUNKSIZE = 200_000

def path_depth(path: str) -> int:
    if not isinstance(path, str):
        return 0
    parts = [p for p in path.split("/") if p]
    return len(parts)

def get_dir(path: str, idx: int):
    if not isinstance(path, str):
        return None
    parts = [p for p in path.split("/") if p]
    return parts[idx] if len(parts) > idx else None

user_rows = []

for chunk in pd.read_json(JSONL_FILE, lines=True, chunksize=CHUNKSIZE):
    if "uid" not in chunk.columns:
        raise ValueError("Missing 'uid' column")

    chunk["uid"] = chunk["uid"].astype(str)
    sub = chunk.loc[chunk["uid"] == UID].copy()
    if sub.empty:
        continue

    sub["time"] = pd.to_datetime(sub["time"], errors="coerce", utc=True)
    sub = sub.dropna(subset=["time"])

    sub["date"] = sub["time"].dt.date.astype(str)
    sub["hour"] = sub["time"].dt.hour

    # params extraction
    if "params" in sub.columns:
        sub["params_path"] = sub["params"].apply(lambda x: x.get("path") if isinstance(x, dict) else None)
        sub["params_nkeys"] = sub["params"].apply(lambda x: len(x) if isinstance(x, dict) else 0)
    else:
        sub["params_path"] = None
        sub["params_nkeys"] = 0

    sub["path_depth"] = sub["params_path"].apply(path_depth)
    sub["dir1"] = sub["params_path"].apply(lambda p: get_dir(p, 0))
    sub["dir2"] = sub["params_path"].apply(lambda p: get_dir(p, 1))

    # boolean masks for type counts (no groupby.apply)
    sub["is_file_accessed"] = (sub["type"] == "file_accessed").astype(int)
    sub["is_login_attempt"] = (sub["type"] == "login_attempt").astype(int)
    sub["is_login_successful"] = (sub["type"] == "login_successful").astype(int)

    user_rows.append(sub[[
        "date","hour","type","params_nkeys","is_file_accessed","is_login_attempt",
        "is_login_successful","params_path","path_depth","dir1","dir2"
    ]])

if not user_rows:
    raise RuntimeError("No rows found for the user; check UID or file path.")

user_df = pd.concat(user_rows, ignore_index=True)

g = user_df.groupby("date", sort=True)

daily_df = pd.DataFrame({
    "events_total": g.size(),
    "active_hours": g["hour"].nunique(),
    "first_hour": g["hour"].min(),
    "last_hour": g["hour"].max(),
    "night_events": g["hour"].apply(lambda s: ((s >= 0) & (s <= 5)).sum()),
    "unique_types": g["type"].nunique(),
    "params_key_count_mean": g["params_nkeys"].mean(),
    "file_events": g["is_file_accessed"].sum(),
    "login_attempt": g["is_login_attempt"].sum(),
    "login_successful": g["is_login_successful"].sum(),
    "unique_paths": g["params_path"].nunique(dropna=True),
    "path_depth_mean": g["path_depth"].mean(),
    "unique_dir1": g["dir1"].nunique(dropna=True),
    "unique_dir2": g["dir2"].nunique(dropna=True),
}).reset_index()

daily_df = daily_df.sort_values("date").reset_index(drop=True)

eps = 1e-9
daily_df["night_fraction"] = daily_df["night_events"] / (daily_df["events_total"] + eps)
daily_df["login_success_rate"] = daily_df["login_successful"] / (daily_df["login_attempt"] + eps)
daily_df["path_reuse_ratio"] = daily_df["file_events"] / (daily_df["unique_paths"] + eps)

print("rows:", len(daily_df), "unique_dates:", daily_df["date"].nunique(),
      "duplicates:", (daily_df["date"].value_counts() > 1).sum())
print(daily_df["date"].min(), daily_df["date"].max())

daily_df.to_csv("daily_features_spectacular-copper-cheetah-postman.csv", index=False)
print("Saved: daily_features_spectacular-copper-cheetah-postman.csv")
print(daily_df.head(10).to_string(index=False))

rows: 88 unique_dates: 88 duplicates: 0
2017-07-07 2017-10-03
Saved: daily_features_spectacular-copper-cheetah-postman.csv
      date  events_total  active_hours  first_hour  last_hour  night_events  unique_types  params_key_count_mean  file_events  login_attempt  login_successful  unique_paths  path_depth_mean  unique_dir1  unique_dir2  night_fraction  login_success_rate  path_reuse_ratio
2017-07-07            77            12           9         23             0             3                    1.0           23             27                27             5         1.155844            1            2        0.000000                 1.0          4.600000
2017-07-08           150            19           0         23            41             3                    1.0           48             51                51             3         1.173333            1            2        0.273333                 1.0         16.000000
2017-07-09           123            18           0         23      

In [ ]:
print(daily_df.shape)

(88, 18)


Checking for duplicates (none)

In [ ]:
print("rows:", len(daily_df), "unique_dates:", daily_df["date"].nunique())
print("duplicates:", (daily_df["date"].value_counts() > 1).sum())
print(daily_df["date"].min(), daily_df["date"].max())

rows: 88 unique_dates: 88
duplicates: 0
2017-07-07 2017-10-03


In [ ]:
print(daily_df.head())

         date  events_total  active_hours  first_hour  last_hour  \
0  2017-07-07            77            12           9         23   
1  2017-07-08           150            19           0         23   
2  2017-07-09           123            18           0         23   
3  2017-07-10           138            18           0         23   
4  2017-07-11           130            20           0         23   

   night_events  unique_types  params_key_count_mean  file_events  \
0             0             3                    1.0           23   
1            41             3                    1.0           48   
2            32             3                    1.0           33   
3            45             3                    1.0           36   
4            39             3                    1.0           34   

   login_attempt  login_successful  unique_paths  path_depth_mean  \
0             27                27             5         1.155844   
1             51                51    

Probabilistic approach for searching flagged days

In [ ]:
import pandas as pd
import numpy as np

INFILE = "daily_features_spectacular-copper-cheetah-postman.csv"
OUTFILE = "anomaly_scores_global_baseline.csv"

df = pd.read_csv(INFILE).sort_values("date")

FEATURES = [
    "events_total",
    "active_hours",
    "night_fraction",
    "unique_types",
    "file_events",
    "login_attempt",
    "login_successful",
    "login_success_rate",
    "unique_paths",
    "path_depth_mean",
    "unique_dir1",
    "unique_dir2",
    "path_reuse_ratio",
]

# Choose threshold: p95 is more sensitive; p99 is stricter
THRESH = "p95"   # change to "p95" if you want more anomalies

eps = 1e-9

# Compute global baselines
baseline = {}
for f in FEATURES:
    s = df[f].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        baseline[f] = {"median": np.nan, "p95": np.nan, "p99": np.nan}
        continue
    baseline[f] = {
        "median": float(np.nanmedian(s)),
        "p95": float(np.nanpercentile(s, 95)),
        "p99": float(np.nanpercentile(s, 99)),
    }

def ratio_to_thresh(x, t):
    # If threshold is 0, use eps to avoid explosion; this keeps ratios interpretable.
    return float(x) / (float(t) + eps)

scores = []
top_explanations = []
n_flags = []

for _, row in df.iterrows():
    contribs = []
    score = 0.0
    flags = 0

    for f in FEATURES:
        x = float(row[f])

        t = baseline[f][THRESH]
        if np.isnan(t):
            continue

        # Flag only if above percentile threshold
        if x > t:
            r = ratio_to_thresh(x, t)
            # contribution: how many "multiples" above threshold (simple and interpretable)
            c = r - 1.0
            score += c
            flags += 1
            contribs.append((f, x, t, r, c))

    contribs.sort(key=lambda z: z[4], reverse=True)
    top = contribs[:5]

    if top:
        expl = "; ".join([f"{f}={x:.3g} vs {THRESH}={t:.3g} ({r:.2f}x)"
                          for (f, x, t, r, _) in top])
    else:
        expl = "No features exceeded " + THRESH

    scores.append(score)
    top_explanations.append(expl)
    n_flags.append(flags)

out = df.copy()
out["anomaly_score"] = scores
out["num_flagged_features"] = n_flags
out["top_contributors"] = top_explanations

# Sort to inspect the most anomalous days
out_sorted = out.sort_values(["anomaly_score", "num_flagged_features"], ascending=False)

out_sorted.to_csv(OUTFILE, index=False)
print(f"Saved: {OUTFILE}")
print(out_sorted[["date", "anomaly_score", "num_flagged_features", "top_contributors"]].head(25).to_string(index=False))

Saved: anomaly_scores_global_baseline.csv
      date  anomaly_score  num_flagged_features                                                                                                                                                                             top_contributors
2017-07-17      23.337159                     4                                    file_events=638 vs p95=53.2 (11.99x); unique_paths=228 vs p95=28.5 (7.99x); events_total=688 vs p95=151 (4.55x); path_depth_mean=7.13 vs p95=2.54 (2.81x)
2017-07-28      15.184428                     4                                     unique_paths=264 vs p95=28.5 (9.25x); file_events=287 vs p95=53.2 (5.39x); path_depth_mean=6.07 vs p95=2.54 (2.39x); events_total=325 vs p95=151 (2.15x)
2017-08-24       2.989261                     4                                      unique_paths=70 vs p95=28.5 (2.45x); file_events=106 vs p95=53.2 (1.99x); path_depth_mean=3.84 vs p95=2.54 (1.51x); events_total=156 vs p95=151 (1.03x)
2017-08-01

### Anomaly Detection with Isolation Forest

Now, let's apply the Isolation Forest algorithm to detect anomalies. Isolation Forest works by randomly selecting a feature and then randomly selecting a split value between the maximum and minimum values of the selected feature. This partitioning continues until data points are isolated. Anomalies are data points that require fewer splits to be isolated. We'll use `scikit-learn` for this.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configuration constants
INFILE = "daily_features_spectacular-copper-cheetah-postman.csv"
CONTAMINATION = 0.05  # Expected anomaly fraction for Isolation Forest

# Load data
df = pd.read_csv(INFILE).sort_values("date").reset_index(drop=True)

# Ensure all features exist and create new_paths_log if missing
for col in ["new_paths_count", "new_paths_log"]:
    if col not in df.columns:
        df[col] = 0.0 # Initialize with 0.0 if not present

if "new_paths_count" in df.columns and "new_paths_log" not in df.columns:
    df["new_paths_log"] = np.log1p(df["new_paths_count"].fillna(0))

# Prepare features: replace inf, impute with median (raw units)
X = df[FEATURES].replace([np.inf, -np.inf], np.nan)
X = X.apply(lambda col: col.fillna(col.median()), axis=0)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest model
iso_model = IsolationForest(
    n_estimators=500,
    contamination=CONTAMINATION,
    random_state=42,
)
iso_model.fit(X_scaled)

# Predict anomaly scores
# sklearn: predict returns 1 for inliers, -1 for outliers
pred = iso_model.predict(X_scaled)

# decision_function: higher = more normal; lower = more anomalous
normality = iso_model.decision_function(X_scaled)
anomaly_score = -normality  # higher = more anomalous

# Create output DataFrame
isolation_forest_df = df[["date"]].copy()
isolation_forest_df["iso_pred"] = pred
isolation_forest_df["iso_anomaly_score"] = anomaly_score

# Save and display top anomalous days
isolation_forest_df_sorted = isolation_forest_df.sort_values("iso_anomaly_score", ascending=False)
isolation_forest_df_sorted.to_csv("isolation_forest_scores.csv", index=False)

print("Saved: isolation_forest_scores.csv")
print("\nTop anomalous days by Isolation Forest:")
display(isolation_forest_df_sorted.head(25))

Saved: isolation_forest_scores.csv

Top anomalous days by Isolation Forest:


,date,iso_pred,iso_anomaly_score
10,2017-07-17,-1,0.170392
21,2017-07-28,-1,0.124234
73,2017-09-18,-1,0.018693
85,2017-10-01,-1,0.009403
8,2017-07-15,-1,0.004657
48,2017-08-24,1,-0.008649
25,2017-08-01,1,-0.016213
32,2017-08-08,1,-0.029341
53,2017-08-29,1,-0.033296
1,2017-07-08,1,-0.033834


### Comparing Anomaly Detection Methods

Let's compare the top anomalous days identified by both the Global Baseline method and the Isolation Forest method.

Checking the consistency between both algorithms (naive probabilistic and isolation forest)

In [ ]:
import pandas as pd

# Load results from Global Baseline method
global_baseline_df = pd.read_csv('anomaly_scores_global_baseline.csv')
# Sort by anomaly_score in descending order
global_baseline_df_sorted = global_baseline_df.sort_values(by='anomaly_score', ascending=False)

# Load results from Isolation Forest method
isolation_forest_df = pd.read_csv('isolation_forest_scores.csv')
# Sort by iso_anomaly_score in descending order
isolation_forest_df_sorted = isolation_forest_df.sort_values(by='iso_anomaly_score', ascending=False)

# Display top N anomalous days for each method
N_COMPARE = 5

print(f"\nTop {N_COMPARE} Anomalous Days by Global Baseline Method for consistent user:")
display(global_baseline_df_sorted[['date', 'anomaly_score', 'top_contributors']].head(N_COMPARE))

print(f"\nTop {N_COMPARE} Anomalous Days by Isolation Forest Method:")
display(isolation_forest_df_sorted[['date', 'iso_anomaly_score']].head(N_COMPARE))


Top 5 Anomalous Days by Global Baseline Method for consistent user:


,date,anomaly_score,top_contributors
0,2017-07-17,23.337159,file_events=638 vs p95=53.2 (11.99x); unique_p...
1,2017-07-28,15.184428,unique_paths=264 vs p95=28.5 (9.25x); file_eve...
2,2017-08-24,2.989261,unique_paths=70 vs p95=28.5 (2.45x); file_even...
3,2017-08-01,2.000000,unique_dir1=3 vs p95=1 (3.00x)
4,2017-07-15,0.848323,login_attempt=62 vs p95=48 (1.29x); login_succ...



Top 5 Anomalous Days by Isolation Forest Method:


,date,iso_anomaly_score
0,2017-07-17,0.170392
1,2017-07-28,0.124234
2,2017-09-18,0.018693
3,2017-10-01,0.009403
4,2017-07-15,0.004657


Searching for an inconsistent user

In [ ]:
"""Finding candidate INCONSISTENT users based on daily volume variability"""

JSONL_FILE = "clue_sample_1GB.jsonl"
CHUNKSIZE = 100_000

DATA_START = pd.Timestamp("2017-07-07")
DATA_END = pd.Timestamp("2017-10-03")  # same window as before
MIN_ACTIVE_DAYS_VAR = 30               # require at least this many days for stability

print("\nStage A: Aggregating per-user, per-day EVENT COUNTS for variability analysis...")

# per_user_per_day_events_var[uid][date] = number of events that day
per_user_per_day_events_var = defaultdict(Counter)

for chunk in pd.read_json(JSONL_FILE, lines=True, chunksize=CHUNKSIZE):
    if not set(["uid", "time"]).issubset(chunk.columns):
        raise ValueError("Chunk missing required columns 'uid' or 'time'.")

    chunk["uid"] = chunk["uid"].astype(str)
    chunk["date"] = pd.to_datetime(chunk["time"], errors="coerce").dt.date
    chunk = chunk.dropna(subset=["date"])

    for (uid, date), group in chunk.groupby(["uid", "date"]):
        per_user_per_day_events_var[uid][date] += len(group)

print("Done aggregating for variability.")
print("Building per-user variability table...")

var_records = []
for uid, date_counts in per_user_per_day_events_var.items():
    days = sorted(date_counts.keys())
    daily_counts = np.array([date_counts[d] for d in days], dtype=float)

    active_days = len(daily_counts)
    if active_days == 0:
        continue

    # Only consider dates in our main window (optional but consistent with earlier analysis)
    # If your earlier aggregation already restricted by date, you can skip this block.
    # Here we keep it simple and use all days present in the sample.
    mean_events = float(daily_counts.mean())
    std_events = float(daily_counts.std(ddof=0))
    cv_events = std_events / (mean_events + 1e-9)  # coefficient of variation

    p95_events = float(np.percentile(daily_counts, 95))
    median_events = float(np.median(daily_counts))
    burst_ratio = p95_events / max(1.0, median_events)

    var_records.append({
        "uid": uid,
        "active_days_var": active_days,
        "mean_events_per_day": mean_events,
        "std_events_per_day": std_events,
        "cv_events": cv_events,
        "median_events_per_day": median_events,
        "p95_events_per_day": p95_events,
        "burst_ratio_events": burst_ratio,
    })

user_variability = pd.DataFrame(var_records)
print("Built variability table with", len(user_variability), "users.")


Stage A: Aggregating per-user, per-day EVENT COUNTS for variability analysis...
Done aggregating for variability.
Building per-user variability table...
Built variability table with 477 users.


Ranking inconsistent users

In [ ]:
"""Ranking candidate INCONSISTENT users (high variability among active users)"""

# Load your previous “top active users” summary
activity_stats = pd.read_csv("top_active_users_chunked.csv")

# Merge variability with activity stats on uid
user_var_merged = user_variability.merge(activity_stats, on="uid", how="inner")

# Filter to users that are reasonably active (same criteria as before, or lighter)
MIN_ACTIVE_DAYS_FOR_INCONSISTENT = 30
MIN_MEDIAN_EVENTS_FOR_INCONSISTENT = 20

candidate_pool = user_var_merged[
    (user_var_merged["active_days"] >= MIN_ACTIVE_DAYS_FOR_INCONSISTENT) &
    (user_var_merged["median_events_per_active_day"] >= MIN_MEDIAN_EVENTS_FOR_INCONSISTENT)
].copy()

# Rank: high coefficient of variation and/or high burst_ratio
candidate_pool["inconsistency_score"] = (
    candidate_pool["cv_events"] +
    0.5 * candidate_pool["burst_ratio_events"]
)

candidate_pool = candidate_pool.sort_values(
    ["inconsistency_score", "active_days", "total_events"],
    ascending=[False, False, False]
)

# Save and show top 20 most inconsistent active users
candidate_pool.to_csv("candidate_inconsistent_users.csv", index=False)
print("Saved: candidate_inconsistent_users.csv")

cols_show = [
    "uid",
    "active_days",
    "coverage",
    "total_events",
    "median_events_per_active_day",
    "p95_events_per_active_day",
    "cv_events",
    "burst_ratio_events",
    "inconsistency_score",
]
cols_show = [c for c in cols_show if c in candidate_pool.columns]

print("\nTop 20 candidate INCONSISTENT users:\n")
print(candidate_pool[cols_show].head(20).to_string(index=False))

Saved: candidate_inconsistent_users.csv

Top 20 candidate INCONSISTENT users:

                                     uid  active_days  coverage  total_events  median_events_per_active_day  p95_events_per_active_day  cv_events  burst_ratio_events  inconsistency_score
     ready-silver-angelfish-quarryworker           83  0.932584         16365                            56                        491   4.175325            8.776786             8.563718
developed-harlequin-falcon-radiodirector           88  0.988764         49985                           108                       1194   2.387527           11.011521             7.893288
        nice-orange-centipede-taxidriver           89  1.000000        248343                          1429                       7784   1.878540            5.447306             4.602192
 shared-fuchsia-cardinal-buildingadvisor           89  1.000000        202615                          1480                       8852   1.352545            5.981486        

Building new features for inconsistent user

In [ ]:
"""Build daily features for the INCONSISTENT user"""

JSONL_FILE = "clue_sample_1GB.jsonl"
CHUNKSIZE = 200_000
INCONSISTENT_UID = "ready-silver-angelfish-quarryworker"  # <- your selected uid

def path_depth(path: str) -> int:
    if not isinstance(path, str):
        return 0
    parts = [p for p in path.split("/") if p]
    return len(parts)

def get_dir(path: str, idx: int):
    if not isinstance(path, str):
        return None
    parts = [p for p in path.split("/") if p]
    return parts[idx] if len(parts) > idx else None

user_rows = []

for chunk in pd.read_json(JSONL_FILE, lines=True, chunksize=CHUNKSIZE):
    if "uid" not in chunk.columns:
        raise ValueError("Missing 'uid' column")

    chunk["uid"] = chunk["uid"].astype(str)
    sub = chunk.loc[chunk["uid"] == INCONSISTENT_UID].copy()
    if sub.empty:
        continue

    sub["time"] = pd.to_datetime(sub["time"], errors="coerce", utc=True)
    sub = sub.dropna(subset=["time"])

    sub["date"] = sub["time"].dt.date.astype(str)
    sub["hour"] = sub["time"].dt.hour

    if "params" in sub.columns:
        sub["params_path"] = sub["params"].apply(lambda x: x.get("path") if isinstance(x, dict) else None)
        sub["params_nkeys"] = sub["params"].apply(lambda x: len(x) if isinstance(x, dict) else 0)
    else:
        sub["params_path"] = None
        sub["params_nkeys"] = 0

    sub["path_depth"] = sub["params_path"].apply(path_depth)
    sub["dir1"] = sub["params_path"].apply(lambda p: get_dir(p, 0))
    sub["dir2"] = sub["params_path"].apply(lambda p: get_dir(p, 1))

    sub["is_file_accessed"] = (sub["type"] == "file_accessed").astype(int)
    sub["is_login_attempt"] = (sub["type"] == "login_attempt").astype(int)
    sub["is_login_successful"] = (sub["type"] == "login_successful").astype(int)

    user_rows.append(sub[[
        "date","hour","type","params_nkeys","is_file_accessed","is_login_attempt",
        "is_login_successful","params_path","path_depth","dir1","dir2"
    ]])

if not user_rows:
    raise RuntimeError(f"No rows found for uid={INCONSISTENT_UID}; check the UID or file path.")

user_df = pd.concat(user_rows, ignore_index=True)
g = user_df.groupby("date", sort=True)

daily_df_inc = pd.DataFrame({
    "events_total": g.size(),
    "active_hours": g["hour"].nunique(),
    "first_hour": g["hour"].min(),
    "last_hour": g["hour"].max(),
    "night_events": g["hour"].apply(lambda s: ((s >= 0) & (s <= 5)).sum()),
    "unique_types": g["type"].nunique(),
    "params_key_count_mean": g["params_nkeys"].mean(),
    "file_events": g["is_file_accessed"].sum(),
    "login_attempt": g["is_login_attempt"].sum(),
    "login_successful": g["is_login_successful"].sum(),
    "unique_paths": g["params_path"].nunique(dropna=True),
    "path_depth_mean": g["path_depth"].mean(),
    "unique_dir1": g["dir1"].nunique(dropna=True),
    "unique_dir2": g["dir2"].nunique(dropna=True),
}).reset_index()

daily_df_inc = daily_df_inc.sort_values("date").reset_index(drop=True)

eps = 1e-9
daily_df_inc["night_fraction"] = daily_df_inc["night_events"] / (daily_df_inc["events_total"] + eps)
daily_df_inc["login_success_rate"] = daily_df_inc["login_successful"] / (daily_df_inc["login_attempt"] + eps)
daily_df_inc["path_reuse_ratio"] = daily_df_inc["file_events"] / (daily_df_inc["unique_paths"] + eps)

outfile_inc = f"daily_features_{INCONSISTENT_UID}.csv"
daily_df_inc.to_csv(outfile_inc, index=False)

print(f"Saved daily features for inconsistent user to: {outfile_inc}")
print(daily_df_inc.head(10).to_string(index=False))
print(daily_df_inc.shape)

Saved daily features for inconsistent user to: daily_features_ready-silver-angelfish-quarryworker.csv
      date  events_total  active_hours  first_hour  last_hour  night_events  unique_types  params_key_count_mean  file_events  login_attempt  login_successful  unique_paths  path_depth_mean  unique_dir1  unique_dir2  night_fraction  login_success_rate  path_reuse_ratio
2017-07-07            12             6           8         21             0             2               1.000000           11              1                 0             7         3.416667            1            4        0.000000            0.000000          1.571429
2017-07-08            15             4           7         16             0             3               1.000000            6              5                 4             2         1.200000            1            2        0.000000            0.800000          3.000000
2017-07-09            14             3          14         22             0             

Running probabilistic approach

In [ ]:
import pandas as pd
import numpy as np

INFILE = "daily_features_ready-silver-angelfish-quarryworker.csv"
OUTFILE = "anomaly_scores_global_baseline_inconsistent.csv"

df = pd.read_csv(INFILE).sort_values("date")

FEATURES = [
    "events_total",
    "active_hours",
    "night_fraction",
    "unique_types",
    "file_events",
    "login_attempt",
    "login_successful",
    "login_success_rate",
    "unique_paths",
    "path_depth_mean",
    "unique_dir1",
    "unique_dir2",
    "path_reuse_ratio",
]

# Choose threshold: p95 is more sensitive; p99 is stricter
THRESH = "p95"   # change to "p95" if you want more anomalies

eps = 1e-9

# Compute global baselines
baseline = {}
for f in FEATURES:
    s = df[f].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        baseline[f] = {"median": np.nan, "p95": np.nan, "p99": np.nan}
        continue
    baseline[f] = {
        "median": float(np.nanmedian(s)),
        "p95": float(np.nanpercentile(s, 95)),
        "p99": float(np.nanpercentile(s, 99)),
    }

def ratio_to_thresh(x, t):
    # If threshold is 0, use eps to avoid explosion; this keeps ratios interpretable.
    return float(x) / (float(t) + eps)

scores = []
top_explanations = []
n_flags = []

for _, row in df.iterrows():
    contribs = []
    score = 0.0
    flags = 0

    for f in FEATURES:
        x = float(row[f])

        t = baseline[f][THRESH]
        if np.isnan(t):
            continue

        # Flag only if above percentile threshold
        if x > t:
            r = ratio_to_thresh(x, t)
            # contribution: how many "multiples" above threshold (simple and interpretable)
            c = r - 1.0
            score += c
            flags += 1
            contribs.append((f, x, t, r, c))

    contribs.sort(key=lambda z: z[4], reverse=True)
    top = contribs[:5]

    if top:
        expl = "; ".join([f"{f}={x:.3g} vs {THRESH}={t:.3g} ({r:.2f}x)"
                          for (f, x, t, r, _) in top])
    else:
        expl = "No features exceeded " + THRESH

    scores.append(score)
    top_explanations.append(expl)
    n_flags.append(flags)

out = df.copy()
out["anomaly_score"] = scores
out["num_flagged_features"] = n_flags
out["top_contributors"] = top_explanations

# Sort to inspect the most anomalous days
out_sorted = out.sort_values(["anomaly_score", "num_flagged_features"], ascending=False)

out_sorted.to_csv(OUTFILE, index=False)
print(f"Saved: {OUTFILE}")
print(out_sorted[["date", "anomaly_score", "num_flagged_features", "top_contributors"]].head(20).to_string(index=False))

Saved: anomaly_scores_global_baseline_inconsistent.csv
      date  anomaly_score  num_flagged_features                                                                                                                                                                                     top_contributors
2017-07-11      72.789631                     6 unique_paths=7.41e+03 vs p95=389 (19.04x); unique_dir2=372 vs p95=19.8 (18.79x); file_events=7.5e+03 vs p95=428 (17.50x); events_total=7.52e+03 vs p95=491 (15.31x); unique_dir1=49 vs p95=7 (7.00x)
2017-08-18      14.173913                     3                                                                                          login_attempt=188 vs p95=23 (8.17x); login_successful=165 vs p95=21 (7.86x); unique_dir1=8 vs p95=7 (1.14x)
2017-07-23       2.571429                     1                                                                                                                                                                   

Using the Isolation forest

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Configuration constants
INFILE = "daily_features_ready-silver-angelfish-quarryworker.csv"
CONTAMINATION = 0.05  # Expected anomaly fraction for Isolation Forest

# Load data
df = pd.read_csv(INFILE).sort_values("date").reset_index(drop=True)

# Ensure all features exist and create new_paths_log if missing
for col in ["new_paths_count", "new_paths_log"]:
    if col not in df.columns:
        df[col] = 0.0 # Initialize with 0.0 if not present

if "new_paths_count" in df.columns and "new_paths_log" not in df.columns:
    df["new_paths_log"] = np.log1p(df["new_paths_count"].fillna(0))

# Prepare features: replace inf, impute with median (raw units)
X = df[FEATURES].replace([np.inf, -np.inf], np.nan)
X = X.apply(lambda col: col.fillna(col.median()), axis=0)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest model
iso_model = IsolationForest(
    n_estimators=500,
    contamination=CONTAMINATION,
    random_state=42,
)
iso_model.fit(X_scaled)

# Predict anomaly scores
# sklearn: predict returns 1 for inliers, -1 for outliers
pred = iso_model.predict(X_scaled)

# decision_function: higher = more normal; lower = more anomalous
normality = iso_model.decision_function(X_scaled)
anomaly_score = -normality  # higher = more anomalous

# Create output DataFrame
isolation_forest_df = df[["date"]].copy()
isolation_forest_df["iso_pred"] = pred
isolation_forest_df["iso_anomaly_score"] = anomaly_score

# Save and display top anomalous days
isolation_forest_df_sorted = isolation_forest_df.sort_values("iso_anomaly_score", ascending=False)
isolation_forest_df_sorted.to_csv("isolation_forest_scores_inconsistent.csv", index=False)

print("Saved: isolation_forest_scores_inconsistent.csv")
print("\nTop anomalous days by Isolation Forest:")
display(isolation_forest_df_sorted.head(20))

Saved: isolation_forest_scores_inconsistent.csv

Top anomalous days by Isolation Forest:


,date,iso_pred,iso_anomaly_score
4,2017-07-11,-1,0.238366
40,2017-08-18,-1,0.080459
10,2017-07-17,-1,0.015138
52,2017-08-30,-1,0.006410
16,2017-07-23,-1,0.003226
14,2017-07-21,1,-0.029034
46,2017-08-24,1,-0.031193
81,2017-10-02,1,-0.039880
3,2017-07-10,1,-0.043432
7,2017-07-14,1,-0.052177


In [ ]:
# Load results from Global Baseline method
global_baseline_df = pd.read_csv('anomaly_scores_global_baseline_inconsistent.csv')
# Sort by anomaly_score in descending order
global_baseline_df_sorted = global_baseline_df.sort_values(by='anomaly_score', ascending=False)

# Load results from Isolation Forest method
isolation_forest_df = pd.read_csv('isolation_forest_scores_inconsistent.csv')
# Sort by iso_anomaly_score in descending order
isolation_forest_df_sorted = isolation_forest_df.sort_values(by='iso_anomaly_score', ascending=False)

# Display top N anomalous days for each method
N_COMPARE = 5

print(f"\nTop {N_COMPARE} Anomalous Days by Global Baseline Method for inconsistent user:")
display(global_baseline_df_sorted[['date', 'anomaly_score', 'top_contributors']].head(N_COMPARE))

print(f"\nTop {N_COMPARE} Anomalous Days by Isolation Forest Method:")
display(isolation_forest_df_sorted[['date', 'iso_anomaly_score']].head(N_COMPARE))


Top 5 Anomalous Days by Global Baseline Method for inconsistent user:


,date,anomaly_score,top_contributors
0,2017-07-11,72.789631,unique_paths=7.41e+03 vs p95=389 (19.04x); uni...
1,2017-08-18,14.173913,login_attempt=188 vs p95=23 (8.17x); login_suc...
2,2017-07-23,2.571429,unique_dir1=25 vs p95=7 (3.57x)
3,2017-08-24,1.622594,events_total=876 vs p95=491 (1.78x); file_even...
4,2017-07-17,1.548852,unique_dir2=28 vs p95=19.8 (1.41x); path_depth...



Top 5 Anomalous Days by Isolation Forest Method:


,date,iso_anomaly_score
0,2017-07-11,0.238366
1,2017-08-18,0.080459
2,2017-07-17,0.015138
3,2017-08-30,0.006410
4,2017-07-23,0.003226


In [ ]:
# Baseline anomaly results (change the filename as needed)
baseline_df = pd.read_csv("anomaly_scores_global_baseline.csv")

# Number of days flagged by at least one feature
num_flagged_baseline = (baseline_df['num_flagged_features'] > 0).sum()
frac_flagged_baseline = (baseline_df['num_flagged_features'] > 0).mean()

print(f"Global Baseline for consistent: {num_flagged_baseline} days flagged out of {len(baseline_df)} ({frac_flagged_baseline:.1%})")

Global Baseline for consistent: 21 days flagged out of 88 (23.9%)


In [ ]:
# Baseline anomaly results (change the filename as needed)
baseline_df = pd.read_csv("anomaly_scores_global_baseline_inconsistent.csv")

# Number of days flagged by at least one feature
num_flagged_baseline = (baseline_df['num_flagged_features'] > 0).sum()
frac_flagged_baseline = (baseline_df['num_flagged_features'] > 0).mean()

print(f"Global Baseline for consistent: {num_flagged_baseline} days flagged out of {len(baseline_df)} ({frac_flagged_baseline:.1%})")

Global Baseline for consistent: 26 days flagged out of 83 (31.3%)


In [ ]:
from google.colab import files

files_to_download = [
    'anomaly_scores_global_baseline.csv',
    'isolation_forest_scores.csv',
    'anomaly_scores_global_baseline_inconsistent.csv',
    'isolation_forest_scores_inconsistent.csv'
]

for file_name in files_to_download:
    files.download(file_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>